# Tavily Search → menu retrieval → OpenAI extraction

Tavily basic Search discovers sources. Python fetches actual restaurant/menu pages and follows daily-menu and PDF links. OpenAI then extracts structured meals from that fetched text, with no OpenAI web search. Python verifies quoted dates, day headings and price evidence before saving the existing app JSON. No Tavily Research or Research output schema is used.

## 1. Load API keys and helpers

Keys stay in `.env`. `OPENAI_MODEL` selects the extraction model (default `gpt-4.1-mini`). Setup reloads the Python modules without making paid requests; rerun it after changing helper code. The workflow functions now live in `menu_pipeline.py`.


In [36]:
import os
import sys
import importlib
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

from dotenv import load_dotenv
from openai import OpenAI

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import tavily_agent
import menu_retrieval
import menu_pipeline as pipeline
importlib.reload(tavily_agent)
importlib.reload(menu_retrieval)
importlib.reload(pipeline)
from tavily_agent import save_menus, recheck_saved_run

load_dotenv(project_root / ".env", override=True)
tavily_key = os.getenv("TAVILY_API_KEY")
extraction_model = os.getenv("OPENAI_MODEL") or "gpt-4.1-mini"
client = OpenAI()
diagnostics_path = project_root / "data/diagnostics/latest-run.json"
print("Tavily key loaded:", bool(tavily_key))
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Extraction model:", extraction_model)


Tavily key loaded: True
OpenAI key loaded: True
Extraction model: gpt-5.4-mini


## 2. Restaurants and date

The notebook and terminal command share `restaurants.json`. Edit names, addresses and stable menu pages there; changing weekly PDF links are discovered automatically. Set `selected_restaurants = restaurants[:1]` to test only Spilka. Saving replaces the file with the selected restaurants, so use the full list for the full app.


In [37]:
restaurants = pipeline.load_restaurants(project_root / "restaurants.json")
selected_restaurants = restaurants  # Use restaurants[:1] to test only Spilka.

today = datetime.now(ZoneInfo("Europe/Bratislava")).date()
print("Requested date:", today)


Requested date: 2026-10-16


## 3. Discover with Tavily basic Search

One basic-search credit per selected restaurant. This cell does not call OpenAI. Search snippets are not treated as menu contents; the next step fetches the sources. Query dates do not establish menu validity.

In [38]:
search_results = pipeline.search_restaurants(selected_restaurants, today, tavily_key)
for result in search_results:
    print(result["restaurant"]["name"], len(result["sources"]), "discovered sources")
    print("Usage:", result["usage"])


Spilka 0 discovered sources
Usage: {'credits': 1}
Centrálna Klubovňa 3 discovered sources
Usage: {'credits': 1}
ALDA 1 discovered sources
Usage: {'credits': 1}
Cafe & Restaurant Michaela 1 discovered sources
Usage: {'credits': 1}


## 4. Retrieve actual menu content

No AI request or Tavily credits here. Python fetches the stable menu page plus relevant search results, follows likely menu links up to two levels, and reads PDF text. It limits each restaurant to eight downloads, 8 MB per download, and 45,000 text characters per source. Blocked pages, JavaScript-only content and image/scanned menus are reported; there is no vision fallback yet. Inspect the URLs and text before extraction. This step does not use stale search snippets as substitute evidence.

In [39]:
retrieved_results = pipeline.retrieve_restaurants(search_results, selected_restaurants, today)
for result in retrieved_results:
    print(result["restaurant"]["name"])
    for source in result["sources"]:
        print(source["kind"], source["url"], len(source["content"]), "characters")
    for error in result["errors"]:
        print("Retrieval issue:", error)


Spilka
html https://www.spilkarestaurant.sk/ponuka/ 1026 characters
pdf https://www.spilkarestaurant.sk/wp-content/uploads/2026/10/261012-SPILKA_MENU_12-16_10_2026.pdf 4799 characters
Centrálna Klubovňa
html https://www.klubovna.sk/ 11918 characters
pdf https://www.klubovna.sk/wp-content/uploads/KLUBOVNA_CK_DENNE_MENU_148x297_WEB-21.pdf 2886 characters
ALDA
html https://www.aldajedalen.sk/ 7784 characters
html https://www.aldajedalen.sk 7784 characters
Cafe & Restaurant Michaela
html https://www.restauraciamichaela.sk/denne-menu 3886 characters


## 5. Extract and validate

OpenAI reads the fetched text without a web-search tool. Python checks the evidence. Matching verified results are reused by default; failed results are retried. Model requests cost tokens; no automatic retries are made.

To intentionally compare another model on every restaurant, change `retry_failed_only` to `False`. Sources and raw responses are checkpointed before local validation. Inspect `menu_results` and the original menus before saving: evidence checks do not prove completeness.


In [40]:
menu_results = pipeline.extract_menus(
    retrieved_results, selected_restaurants, today,
    client=client,
    model=extraction_model,
    diagnostics_path=diagnostics_path,
    previous_results=globals().get("menu_results", []),
    retry_failed_only=True,
)
for result in menu_results:
    print(result["restaurant"]["name"], result["status"], len(result["meals"]), "meals")
    print(result.get("reason") or result["notes"])


Spilka verified 6 meals
['Requested date falls on the dated weekly menu 12. 10. – 16. 10. 2026; Friday is explicitly listed.', 'The PDF states weekly specials are served all week and outside daily-menu time, so they are included for Friday.', 'No with-meal surcharge beyond the listed 1,99 € soup price was stated; the menu text does not explicitly mark soup as included in any bundle.']
Centrálna Klubovňa verified 3 meals
['Requested date falls within the dated menu week 12.10.–16.10.2026; year is explicit in the source.', 'The PDF states one daily soup and a standalone soup price, but the with-meal surcharge is not explicitly given for the soup itself; additional_price_eur is therefore set to 0 only because the menu says the meal comes with soup, while the standalone price is 1,50 €.', 'Staročeská zemiaková polievka: unsupported free-soup claim removed; surcharge is unknown.']
ALDA verified 8 meals
[]
Cafe & Restaurant Michaela verified 4 meals
['Menu page contains multiple dated weekda

## 6. Save verified meals

No API requests. ALDA may be omitted while its current menu is unavailable; it contributes no meals, and old ALDA meals are not carried forward. Every other selected restaurant must pass validation. Remove ALDA from `allowed_empty_restaurants` to require all restaurants again. Refresh the app after saving.


In [41]:
# Explicit exception for ALDA while its current menu is unavailable.
allowed_empty_restaurants = {"ALDA"}
omitted = {
    record["restaurant"]["name"] for record in menu_results
    if record["restaurant"]["name"] in allowed_empty_restaurants
    and record.get("status") == "needs_review" and not record.get("meals")
}
records_to_save = [r for r in menu_results if r["restaurant"]["name"] not in omitted]
restaurants_to_save = [r for r in selected_restaurants if r["name"] not in omitted]
menus_data = save_menus(records_to_save, restaurants_to_save, today, project_root / "menus.json")
print("Saved", len(menus_data["meals"]), "meals")
if omitted:
    print("No meals saved for:", ", ".join(sorted(omitted)))


Saved 21 meals


## Optional: recheck saved responses

After changing validation code, run setup to reload helpers, then this cell. No API requests and no menus.json write. It updates `menu_results` in memory for review and saving.


In [42]:
menu_results = recheck_saved_run(diagnostics_path)
for result in menu_results:
    print(result["restaurant"]["name"], result["status"], len(result["meals"]), "meals")
    print(result.get("reason") or result["notes"])


Spilka verified 6 meals
['Requested date falls on the dated weekly menu 12. 10. – 16. 10. 2026; Friday is explicitly listed.', 'The PDF states weekly specials are served all week and outside daily-menu time, so they are included for Friday.', 'No with-meal surcharge beyond the listed 1,99 € soup price was stated; the menu text does not explicitly mark soup as included in any bundle.']
Centrálna Klubovňa verified 3 meals
['Requested date falls within the dated menu week 12.10.–16.10.2026; year is explicit in the source.', 'The PDF states one daily soup and a standalone soup price, but the with-meal surcharge is not explicitly given for the soup itself; additional_price_eur is therefore set to 0 only because the menu says the meal comes with soup, while the standalone price is 1,50 €.', 'Staročeská zemiaková polievka: unsupported free-soup claim removed; surcharge is unknown.']
ALDA verified 8 meals
[]
Cafe & Restaurant Michaela verified 4 meals
['Menu page contains multiple dated weekda

## Optional: recover after local processing fails

If a paid response returned but local processing failed, reload helpers in setup and run this cell instead of repeating extraction. It reads the checkpoint from disk, so a kernel restart does not lose the paid response. It makes no API requests and does not write menus.json. Resume the extraction cell afterward if other restaurants remain unprocessed.


In [ ]:
menu_results = pipeline.recover_extraction(diagnostics_path)
for result in menu_results:
    print(result["restaurant"]["name"], result["status"], len(result["meals"]), "meals")
    print(result.get("reason") or result["notes"])
